In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10.95})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result, evaluate_model_prediction, evaluate_model_rollout,
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

---

## update df:

In [ ]:
from admpe.data_management import QUANTITATIVE_RESULTS_ROOT

In [ ]:
# exp_name = "noiseless_w_obj"
exp_name = "noise_1_e_minus_6"

In [ ]:
modeling_results_paths = list_modeling_results(
    system_name=None,
    model_type="neural_euler",
    status="completed",
    exp_name=exp_name,
    excitation_objective=["dmpe_uniform", "dmpe_static_dynamics", "dmpe_static_mixture_dynamics"],
    
)
print(len(modeling_results_paths))
modeling_results_paths

In [ ]:
name = f"modeling_results_{exp_name}"

# if (QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet").exists():
#     modeling_results = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")
# else:
#     modeling_results = None

modeling_results = None

modeling_results = update_modeling_df(modeling_results, modeling_results_paths)
modeling_results.to_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")
print("Updated df stored to: ", QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")

In [ ]:
modeling_results

In [ ]:
modeling_results = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")
modeling_results

## Model performance by excitation objective

Each system has one figure with RMSE, MAE and WCE panels. Boxes group individual runs by model and excitation objective; lower errors are better. The sample counts exclude missing or infinite metric values. Logarithmic axes make the wide error ranges easier to compare; set `log_scale=False` for linear axes.

In [ ]:
from admpe.evaluation.modeling_visualization import visualize_model_prediction_metric_boxplots, visualize_model_rollout_metric_boxplots

In [ ]:
metric_boxplots = visualize_model_prediction_metric_boxplots(modeling_results, log_scale=True, figure_width=figurewidth_in)
figure_root = QUANTITATIVE_RESULTS_ROOT / "modeling_figs" / "boxplots"
figure_root.mkdir(parents=True, exist_ok=True)

for system_name, (fig, axes) in metric_boxplots.items():
    fig.savefig(figure_root / f"{name}_boxplots_pred_{system_name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(figure_root / f"{name}_boxplots_pred_{system_name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

print("Figures saved to:", figure_root)

In [ ]:
rollout_boxplots = visualize_model_rollout_metric_boxplots(
    modeling_results,
    log_scale=True,
    figure_width=figurewidth_in,
    hide_outliers=True
)
figure_root = QUANTITATIVE_RESULTS_ROOT / "modeling_figs" / "boxplots"
figure_root.mkdir(parents=True, exist_ok=True)

for (system_name, signal_name), (fig, axes) in rollout_boxplots.items():
    fig.savefig(figure_root / f"{name}_boxplots_rollout_{signal_name}_{system_name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(figure_root / f"{name}_boxplots_rollout_{signal_name}_{system_name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)
# print("Figures saved to:", figure_root)